In [ ]:
import argopy

In [ ]:
# Define your space-time box: [lon_min, lon_max, lat_min, lat_max, date_min, date_max]
box = [-170, -135, 40, 50, "2020-01", "2021-01"]
# Load the BGC index and search
idx = argopy.ArgoIndex(index_file="bgc-s").load()
npac_doxy = (
    idx
    .query.box(box)
    .query.params('DOXY')
)

# View results as a DataFrame
df = idx.to_dataframe()
print(df[["file", "latitude", "longitude", "date", "wmo"]].head())

In [ ]:
# Fetch data 100-1000 m depth for the DOXY parameter
with argopy.set_options(ds_profiles='bgc', src='gdac', mode='research'):
    params = 'DOXY'  # eg: 'DOXY' or ['DOXY', 'BBP700']
    f = argopy.DataFetcher(params=params)
    f = f.region([-170, -135, 40, 50, 100, 1000, "2021-01", "2021-06"])
    f.load()


In [ ]:
ds_points = f.data
ds_profiles = ds_points.argo.point2profile()
print(ds_profiles)


In [ ]:
import gsw

# 1. Calculate Absolute Salinity (SA)
SA = gsw.SA_from_SP(ds_profiles.PSAL, ds_profiles.PRES, ds_profiles.LONGITUDE, ds_profiles.LATITUDE)

# 2. Calculate Conservative Temperature (CT)
CT = gsw.CT_from_t(SA, ds_profiles.TEMP, ds_profiles.PRES)

# 3. Calculate In-situ Density (rho)
ds_profiles['DENSITY'] = gsw.rho(SA, CT, ds_profiles .PRES)

# Optional: Calculate Sigma-theta (Potential Density Anomaly)
# This is often what researchers actually plot (rho - 1000)
ds_profiles['SIGMA_THETA'] = gsw.sigma0(SA, CT)
print(ds_profiles)

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 6), sharey=False)

for i in range(ds_profiles.N_PROF.size):
    doxy = ds_profiles['DOXY'].isel(N_PROF=i)
    pres = ds_profiles['PRES'].isel(N_PROF=i)
    sigma_theta = ds_profiles['SIGMA_THETA'].isel(N_PROF=i)
    label = f'Float {i+1}'
    
    # Panel 1 and 2: DO vs Depth/Density
    ax1.plot(doxy, pres, label=label)
    ax2.plot(doxy, sigma_theta, label=label)
    
# Invert pressure axis (deeper = down)
ax1.invert_yaxis()
ax1.set_xlabel('DOXY (µmol/kg)')
ax1.set_ylabel('Pressure (dbar)')
ax1.set_title('DOXY vs Depth')

ax2.invert_yaxis()
ax2.set_xlabel('DOXY (µmol/kg)')
ax2.set_ylabel('σ₀ (kg/m³)')
ax2.set_title('DOXY vs Density')

ax1.legend()
plt.tight_layout()
plt.show()